# Validación estadística de los clusters

Con muestras grandes todo p-value sale aproximadamente cero. Por eso el reporte se centra en effect size, que es lo que distingue diferencias triviales de diferencias importantes.

**Autor:** Andrés Fernando Gómez Rojas

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import scikit_posthocs as sp
from src.statistical_tests import (test_normality_by_cluster,
                                    run_omnibus_numerical,
                                    run_omnibus_categorical,
                                    manova_test)

df = pd.read_pickle('../data/processed/df_clustered.pkl')
num_vars = ['n_sessions', 'totals.hits', 'totals.pageviews',
            'hits_per_session', 'pageviews_per_session',
            'bounce_prop', 'weekend_prop', 'hour']
cat_vars = ['channelGrouping', 'device.deviceCategory', 'os_top', 'part_of_day']
pd.set_option('display.float_format', '{:.3f}'.format)

## Test de normalidad

Aplicamos el test de D'Agostino-Pearson en cada variable dentro de cada cluster. Si alguna variable no es normal en algún cluster, descartamos ANOVA paramétrico y usamos Kruskal-Wallis.

In [2]:
normalidad = test_normality_by_cluster(df, num_vars)
pd.Series(normalidad, name='normal_en_todos_los_clusters').to_frame()

,normal_en_todos_los_clusters
n_sessions,False
totals.hits,False
totals.pageviews,False
hits_per_session,False
pageviews_per_session,False
bounce_prop,False
weekend_prop,False
hour,False


Ninguna variable es normal en todos los clusters. La decisión es Kruskal-Wallis.

## Test omnibus con effect size

Kruskal-Wallis prueba si al menos un cluster es diferente del resto. Eta-cuadrado mide la magnitud de la diferencia: 0.01 es pequeña, 0.06 mediana, 0.14 grande.

In [3]:
resultados_num = run_omnibus_numerical(df, num_vars)
resultados_num

,variable,estadistico_H,p_value,eta_cuadrado,magnitud
3,hits_per_session,6451.191,0.000,0.645,grande
4,pageviews_per_session,6380.251,0.000,0.638,grande
0,n_sessions,5613.941,0.000,0.562,grande
2,totals.pageviews,5389.801,0.000,0.539,grande
1,totals.hits,5283.579,0.000,0.528,grande
6,weekend_prop,5027.435,0.000,0.503,grande
5,bounce_prop,2570.251,0.000,0.257,grande
7,hour,174.625,0.000,0.017,pequeno


Siete de ocho variables muestran effect size grande. `hits_per_session` y `pageviews_per_session` lideran con eta-cuadrado por encima de 0.6, lo que confirma que los clusters capturan diferencias enormes en intensidad de engagement por sesión.

La excepción es `hour`: eta-cuadrado por debajo de 0.02. La hora del día no separa los clusters. Esto contradice supuestos comunes en marketing de que el momento de la visita es un segmentador fuerte, y vale la pena explicitarlo.

## Post-hoc de Dunn con corrección de Bonferroni

El test omnibus dice que al menos un cluster difiere; el post-hoc identifica cuáles pares específicos difieren. Bonferroni corrige por las múltiples comparaciones para mantener el error tipo I controlado.

In [4]:
for var in ['hits_per_session', 'weekend_prop']:
    posthoc = sp.posthoc_dunn(df, val_col=var, group_col='cluster', p_adjust='bonferroni')
    posthoc.index = ['C0', 'C1', 'C2', 'C3']
    posthoc.columns = ['C0', 'C1', 'C2', 'C3']
    sig = posthoc.map(lambda x: '***' if x < 0.001 else '**' if x < 0.01 else '*' if x < 0.05 else 'ns')
    print(f'\n>>> {var}')
    print(sig)


>>> hits_per_session
     C0   C1   C2   C3
C0   ns  ***   ns  ***
C1  ***   ns  ***  ***
C2   ns  ***   ns  ***
C3  ***  ***  ***   ns

>>> weekend_prop
     C0   C1   C2   C3
C0   ns  ***  ***  ***
C1  ***   ns  ***  ***
C2  ***  ***   ns  ***
C3  ***  ***  ***   ns


Hallazgo importante en `hits_per_session`: C0 y C2 muestran `ns` (no significativo). Estadísticamente, Desktop Engaged y Weekend Warriors tienen el mismo perfil de intensidad por sesión. Su única diferencia real está en `weekend_prop`, donde sí difieren con significancia máxima.

Interpretación: C2 no es un tipo de usuario distinto de C0; es el mismo tipo de usuario activo en momentos diferentes. Esta es la clase de matiz que el clustering por sí solo no expone y que solo emerge al aplicar tests formales sobre los resultados.

## Tests para variables categóricas

Chi-cuadrado prueba independencia entre la variable categórica y el cluster; Cramer's V mide la fuerza de la asociación. V por debajo de 0.1 es trivial, 0.3 moderado, 0.5 fuerte.

In [5]:
resultados_cat = run_omnibus_categorical(df, cat_vars)
resultados_cat

,variable,chi2,p_value,cramers_v,fuerza
1,device.deviceCategory,9841.561,0.000,0.702,fuerte
2,os_top,9789.531,0.000,0.571,fuerte
0,channelGrouping,1084.848,0.000,0.190,debil
3,part_of_day,492.598,0.000,0.128,debil


`device.deviceCategory` tiene Cramer's V = 0.70, asociación fuerte. El sistema operativo también, lo cual es esperable porque está correlacionado con el dispositivo. En contraste, `channelGrouping` apenas separa con V = 0.19. Esto vuelve a contradecir el supuesto típico de que el canal es el segmentador principal: en este dataset el dispositivo manda.

## MANOVA: prueba multivariada global

Wilks' Lambda prueba si los centroides multivariados de los clusters difieren en el espacio conjunto de todas las variables. Es el equivalente multivariado del test omnibus.

In [6]:
manova_vars = ['n_sessions', 'totals.hits', 'totals.pageviews',
               'hits_per_session', 'pageviews_per_session',
               'bounce_prop', 'weekend_prop']
print(manova_test(df, manova_vars))

                   Multivariate linear model
                                                                
----------------------------------------------------------------
       Intercept        Value  Num DF   Den DF   F Value  Pr > F
----------------------------------------------------------------
          Wilks' lambda 0.3792 7.0000 9988.0000 2336.1236 0.0000
         Pillai's trace 0.6208 7.0000 9988.0000 2336.1236 0.0000
 Hotelling-Lawley trace 1.6373 7.0000 9988.0000 2336.1236 0.0000
    Roy's greatest root 1.6373 7.0000 9988.0000 2336.1236 0.0000
----------------------------------------------------------------
                                                                
----------------------------------------------------------------
         cluster         Value  Num DF   Den DF  F Value  Pr > F
----------------------------------------------------------------
           Wilks' lambda 0.6178 7.0000 9988.0000 882.7517 0.0000
          Pillai's trace 0.3822 7.0000 9988.0

Wilks' Lambda muy por debajo de uno con p prácticamente cero confirma que los centroides multivariados de los cuatro clusters son globalmente distintos. Es la confirmación final de que los grupos son entidades separadas en el espacio conjunto, no solo en variables individuales.